# ASQ-PHI Pipeline: LLM-Based Text De-Identification

Text track of the project *Benchmarking LLMs and VLMs for Automated De-Identification of Clinical Text and Medical Images*.

| Section | Project step | Output |
|---|---|---|
| 1. Setup | – | imports, config |
| 2. Preprocessing | Step 1 | `data/asq_phi.jsonl`, `data/taxonomy_mapping.json` |
| 3. Output schema | Step 2 | Pydantic model used for constrained decoding |
| 4. Prompts | Step 2 | system prompt + few-shot examples |
| 5. Model & generation | Step 2 | Qwen3-8B with constrained JSON decoding (non-thinking / thinking) |
| 6. Post-processing | Step 2 | parsed spans with character offsets |
| 7. Run conditions | Step 2 | `predictions/qwen3-8b_<condition>.jsonl`, incl. the Presidio baseline |
| 8. Evaluation | Step 4 (text side) | `results/*.csv` |
| 9. Error analysis | Step 4 (text side) | false negatives / false positives / label confusions |

Steps 3 (VLM on MIDI-B), 5 (cross-modality comparison) and 6 (report) are outside this notebook. The files saved in `results/` are what Step 5 compares against the image track.

**Hardware:** sections 5 and 7 need a GPU with about 18 GB of memory for Qwen3-8B in bfloat16 (Colab). All other sections run on a laptop. To test the whole pipeline on a Mac, set `MODEL_ID = "Qwen/Qwen3-0.6B"` and `LIMIT = 5`.

**Re-running:** finished prediction files are reused. The LLM is only loaded, and Presidio only run, for queries that have no prediction yet, so the whole notebook can be re-run without repeating the slow parts.

## 1. Setup

Install the dependencies from requirements.txt

In [ ]:
import json, re, hashlib, bisect, random, time
from pathlib import Path
from collections import Counter
from enum import Enum

import pandas as pd
from pydantic import BaseModel, ValidationError
from tqdm.auto import tqdm

# paths
RAW_PATH      = Path("ASQ/synthetic_clinical_queries.txt")
TAXONOMY_PATH = Path("ASQ/asq_taxonomy.txt")
DATA_PATH     = Path("data/asq_phi.jsonl")
MAPPING_PATH  = Path("data/taxonomy_mapping.json")
PRED_DIR      = Path("predictions")
RESULTS_DIR   = Path("results")
for d in (DATA_PATH.parent, PRED_DIR, RESULTS_DIR):
    d.mkdir(exist_ok=True)

# llm model
MODEL_ID  = "Qwen/Qwen3-0.6B"    # local test
MODEL_TAG = MODEL_ID.split("/")[-1].lower()
SEED = 42

# run size 
LIMIT = None   # None uses the full dataset
THINK_SUBSET = None   # thinking condition on a fixed random subset, also uses full dataset

CONDITIONS = {
    "zeroshot":      {"fewshot": False, "think": False},
    "fewshot":       {"fewshot": True,  "think": False},
    "fewshot_think": {"fewshot": True,  "think": True},
    "presidio":      {"fewshot": False, "think": False}}   # non-LLM baseline (Section 7.3)

RUN = ["zeroshot", "fewshot"]   # LLM conditions to run; add "fewshot_think" for the ablation

## 2. Preprocessing

### 2.1 Parse the ASQ text file

Each block in the file is structured as:
```
===QUERY===
<query text>
===PHI_TAGS===
{"identifier_type": "NAME", "value": "Anna S."}
```
We create IDs from each block's position and find the offsets of each gold value in the text (Section 2.3).

In [2]:
QUERY_MARK = "===QUERY==="
TAGS_MARK  = "===PHI_TAGS==="

TAXONOMY = []
with open(TAXONOMY_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line != "":
            TAXONOMY.append(line)

def parse_asq_txt(path):
    raw = path.read_text(encoding="utf-8")
    blocks = raw.split(QUERY_MARK)
    assert blocks[0].strip() == "", "unexpected content before first ===QUERY==="
    records = []
    for i, block in enumerate(blocks[1:]):
        assert TAGS_MARK in block, f"block {i} has no {TAGS_MARK}"
        query_part, tags_part = block.split(TAGS_MARK, 1)
        text = query_part.strip("\n")
        gold = []
        for line in tags_part.splitlines():
            if line.strip():
                tag = json.loads(line)
                gold.append({"asq_type": tag["identifier_type"], "text": tag["value"]})
        records.append({"id": f"asq_{i:04d}", "text": text, "gold": gold})
    return records

data = parse_asq_txt(RAW_PATH)
print(len(data), "queries;", "taxonomy:", TAXONOMY)

1051 queries; taxonomy: ['GEOGRAPHIC_LOCATION', 'NAME', 'DATE', 'MEDICAL_RECORD_NUMBER', 'HEALTH_PLAN_BENEFICIARY_NUMBER', 'PHONE_NUMBER', 'SOCIAL_SECURITY_NUMBER', 'EMAIL_ADDRESS', 'UNIQUE_IDENTIFIER', 'ACCOUNT_NUMBER', 'FAX_NUMBER', 'CERTIFICATE_LICENSE_NUMBER', 'IP_ADDRESS']


### 2.2 Shared taxonomy

ASQ-PHI's 13 categories are mapped onto 5 top-level categories, which the image track (MIDI-B) will also use. The model predicts these shared labels.

Merging the identifier types also helps with label noise in ASQ: the same string `123-45-6789` is tagged as SOCIAL_SECURITY_NUMBER in some queries and MEDICAL_RECORD_NUMBER in others.

> This might change depending on the taxonomy on the image data available

In [3]:
SHARED_TAXONOMY = {
    "NAME":     ["NAME"],
    "DATE":     ["DATE"],
    "LOCATION": ["GEOGRAPHIC_LOCATION"],
    "CONTACT":  ["PHONE_NUMBER", "FAX_NUMBER", "EMAIL_ADDRESS", "IP_ADDRESS"],
    "ID":       ["MEDICAL_RECORD_NUMBER", "HEALTH_PLAN_BENEFICIARY_NUMBER", "SOCIAL_SECURITY_NUMBER",
                 "UNIQUE_IDENTIFIER", "ACCOUNT_NUMBER", "CERTIFICATE_LICENSE_NUMBER"],
}

ASQ_TO_SHARED = {}
for shared_label, asq_types in SHARED_TAXONOMY.items():
    for asq_type in asq_types:
        ASQ_TO_SHARED[asq_type] = shared_label

unmapped = set(TAXONOMY) - set(ASQ_TO_SHARED)
assert not unmapped, f"ASQ categories without a shared label: {unmapped}"

with MAPPING_PATH.open("w", encoding="utf-8") as f:
    json.dump(SHARED_TAXONOMY, f, indent=2)

pd.DataFrame([(s, a) for s, types in SHARED_TAXONOMY.items() for a in types],
             columns=["shared_label", "asq_type"])

,shared_label,asq_type
0,NAME,NAME
1,DATE,DATE
2,LOCATION,GEOGRAPHIC_LOCATION
3,CONTACT,PHONE_NUMBER
4,CONTACT,FAX_NUMBER
5,CONTACT,EMAIL_ADDRESS
6,CONTACT,IP_ADDRESS
7,ID,MEDICAL_RECORD_NUMBER
8,ID,HEALTH_PLAN_BENEFICIARY_NUMBER
9,ID,SOCIAL_SECURITY_NUMBER


### 2.3 Gold offsets, shared labels, and saving the benchmark file

For each gold value we find where it starts and ends in the query (its character offsets). Evaluation compares predicted and gold spans by these positions.

- **Standalone matches first.** A short value such as `UCSF` can also appear inside an ID such as `UCSF-12345`, so a match surrounded by letters, digits or `-` is only used if there is no other.
- **Quote normalisation.** Curly quotes are compared as straight quotes, so `Children's Clinic` matches `Children’s Clinic`. Both are one character, so the offsets stay valid for the original text.

In [ ]:
QUOTES = str.maketrans({"\u2019": "'", "\u2018": "'", "\u201c": '"', "\u201d": '"'})

def locate(text, value):
    t, v = text.translate(QUOTES), value.translate(QUOTES)
    m = re.search(rf"(?<![\w-]){re.escape(v)}(?![\w-])", t) # standalone occurrence first
    if m:
        return m.start(), m.end()
    i = t.find(v)         # fallback: any occurrence
    return (i, i + len(v)) if i >= 0 else (None, None)

for r in data:
    for g in r["gold"]:
        g["label"] = ASQ_TO_SHARED[g["asq_type"]]
        g["start"], g["end"] = locate(r["text"], g["text"])

with DATA_PATH.open("w", encoding="utf-8") as f:
    for r in data:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

DATA_SHA = hashlib.sha1(DATA_PATH.read_bytes()).hexdigest()[:8]

all_gold = [g for r in data for g in r["gold"]]
print(f"{len(data)} queries, {len(all_gold)} gold spans, "
      f"{sum(not r['gold'] for r in data)} queries without PHI  (data version {DATA_SHA})")
print("shared label counts:", Counter(g["label"] for g in all_gold).most_common())
print("ASQ type counts:    ", Counter(g["asq_type"] for g in all_gold).most_common())
print("labels not in taxonomy:", {g["asq_type"] for g in all_gold} - set(TAXONOMY))
print("gold values not found in text:",
      [(r["id"], g["text"]) for r in data for g in r["gold"] if g["start"] is None])
print("matched only after quote normalisation:",
      [(r["id"], g["text"], r["text"][g["start"]:g["end"]]) for r in data for g in r["gold"]
       if g["start"] is not None and r["text"][g["start"]:g["end"]] != g["text"]])
data[1]

1051 queries, 2973 gold spans, 219 queries without PHI  (data version 5164b5d0)
shared label counts: [('LOCATION', 826), ('NAME', 814), ('DATE', 806), ('ID', 448), ('CONTACT', 79)]
ASQ type counts:     [('GEOGRAPHIC_LOCATION', 826), ('NAME', 814), ('DATE', 806), ('MEDICAL_RECORD_NUMBER', 305), ('HEALTH_PLAN_BENEFICIARY_NUMBER', 91), ('PHONE_NUMBER', 45), ('SOCIAL_SECURITY_NUMBER', 33), ('EMAIL_ADDRESS', 31), ('UNIQUE_IDENTIFIER', 14), ('ACCOUNT_NUMBER', 4), ('FAX_NUMBER', 2), ('CERTIFICATE_LICENSE_NUMBER', 1), ('IP_ADDRESS', 1)]
labels not in taxonomy: set()
gold values not found in text: []
matched only after quote normalisation: [('asq_0149', "Children's Clinic", 'Children’s Clinic')]


{'id': 'asq_0001',
 'text': "Evaluation of long-term outcomes for bypass surgery in patients over 60, referencing Mr. James T., operated at St. Vincent's on May 30th, 2022 (ID: 987654321)?",
 'gold': [{'asq_type': 'NAME',
   'text': 'James T.',
   'label': 'NAME',
   'start': 89,
   'end': 97},
  {'asq_type': 'GEOGRAPHIC_LOCATION',
   'text': "St. Vincent's",
   'label': 'LOCATION',
   'start': 111,
   'end': 124},
  {'asq_type': 'DATE',
   'text': 'May 30th, 2022',
   'label': 'DATE',
   'start': 128,
   'end': 142},
  {'asq_type': 'UNIQUE_IDENTIFIER',
   'text': '987654321',
   'label': 'ID',
   'start': 148,
   'end': 157}]}

## 3. Output schema

This one Pydantic model is used three ways:
1. its JSON schema constrains decoding, so the output always parses;
2. it validates the raw output;
3. it builds the few-shot answers.

`type` is an Enum, so the model can only produce shared-taxonomy labels.

In [5]:
PHIType = Enum("PHIType", {label: label for label in SHARED_TAXONOMY}, type=str)

class Span(BaseModel):
    type: PHIType
    text: str

class Extraction(BaseModel):
    spans: list[Span]

print(json.dumps(Extraction.model_json_schema(), indent=2))

{
  "$defs": {
    "PHIType": {
      "enum": [
        "NAME",
        "DATE",
        "LOCATION",
        "CONTACT",
        "ID"
      ],
      "title": "PHIType",
      "type": "string"
    },
    "Span": {
      "properties": {
        "type": {
          "$ref": "#/$defs/PHIType"
        },
        "text": {
          "title": "Text",
          "type": "string"
        }
      },
      "required": [
        "type",
        "text"
      ],
      "title": "Span",
      "type": "object"
    }
  },
  "properties": {
    "spans": {
      "items": {
        "$ref": "#/$defs/Span"
      },
      "title": "Spans",
      "type": "array"
    }
  },
  "required": [
    "spans"
  ],
  "title": "Extraction",
  "type": "object"
}


## 4. Prompts

### 4.1 System prompt

The rules follow the ASQ-PHI annotation conventions found in the data:
- titles are usually part of a name (`Dr. Lopez`: 90 cases with, 35 without);
- field labels are not part of an identifier (`MRN: SMH-54321` → `SMH-54321`);
- hospitals and clinics are locations;
- a year on its own (`diagnosed in 2019`), ages and ethnicity are never tagged.

**Relative dates** (`last week`) are tagged in some ASQ queries and not in others. We tell the model to tag them because the project puts recall first; this may cost a few false positives.

In [6]:
LABEL_DEFINITIONS = {
    "NAME":     "names of patients, doctors, relatives or other people, including partial names, initials and nicknames "
                "(e.g. 'Anna S.', 'Dr. Lopez', 'Bobby'). Keep a title such as 'Dr.' or 'Mr.' if it directly precedes the name.",
    "DATE":     "calendar dates tied to a person or event, in any format, including partial and relative dates "
                "(e.g. 'April 12, 2023', 'Feb 22nd', \"Nov 11th '23\", '04/12/2022', 'last Friday').",
    "LOCATION": "named places: hospitals, clinics, medical centers, cities, states, streets, addresses "
                "(e.g. 'Mayo Clinic in Rochester, MN', 'Elm Clinic', 'San Francisco').",
    "CONTACT":  "phone numbers, fax numbers, email addresses, IP addresses, URLs.",
    "ID":       "medical record numbers, health plan / insurance numbers, social security numbers, account numbers, "
                "licence or certificate numbers, and any other patient, case or record identifier.",
}

SYSTEM = f"""You are a clinical de-identification system. Extract every mention of protected health information (PHI) from the user's text.

Categories:
{chr(10).join(f"- {label}: {definition}" for label, definition in LABEL_DEFINITIONS.items())}

Rules:
1. Copy each span VERBATIM from the text: same spelling, casing and punctuation. Never paraphrase or normalise.
2. List every occurrence in order of appearance, even if the same string repeats.
3. Do not include field labels in a span: for "MRN: SMH-54321" return "SMH-54321"; for "Fax: 650-123-4567" return "650-123-4567".
4. NOT PHI: ages ("34-year-old"), sex, ethnicity, diseases, drugs, lab values, a year on its own ("diagnosed in 2019"),
   generic departments or regions ("cardiology department", "in the south"), and medical eponyms ("Graves' disease").
5. If there is no PHI, return {{"spans": []}}.

Return only JSON of the form {{"spans": [{{"type": "<CATEGORY>", "text": "<exact substring>"}}]}}."""

print(SYSTEM)

You are a clinical de-identification system. Extract every mention of protected health information (PHI) from the user's text.

Categories:
- NAME: names of patients, doctors, relatives or other people, including partial names, initials and nicknames (e.g. 'Anna S.', 'Dr. Lopez', 'Bobby'). Keep a title such as 'Dr.' or 'Mr.' if it directly precedes the name.
- DATE: calendar dates tied to a person or event, in any format, including partial and relative dates (e.g. 'April 12, 2023', 'Feb 22nd', "Nov 11th '23", '04/12/2022', 'last Friday').
- LOCATION: named places: hospitals, clinics, medical centers, cities, states, streets, addresses (e.g. 'Mayo Clinic in Rochester, MN', 'Elm Clinic', 'San Francisco').
- CONTACT: phone numbers, fax numbers, email addresses, IP addresses, URLs.
- ID: medical record numbers, health plan / insurance numbers, social security numbers, account numbers, licence or certificate numbers, and any other patient, case or record identifier.

Rules:
1. Copy each spa

### 4.2 Few-shot examples

These examples are **written by hand, not taken from ASQ-PHI**, so no test data leaks into the prompt. They follow ASQ's style and cover:
1. **easy:** a full, well-formed query;
2. **tricky:** a nickname, a relative date, a short year (`'23`), a partial address, a title, an ID with a field label, contact details;
3. **hard negative:** age, a year on its own and eponyms, with no PHI.

The `assert` checks that every example span appears verbatim in its text, so the examples follow Rule 1.

In [7]:
FEWSHOT = [
    # 1) easy
    ("Dosing of apixaban for a 72-year-old male, Peter W., admitted to Riverside General Hospital "
     "on June 3rd, 2023 (MRN: RGH-445120)?",
     Extraction(spans=[
         Span(type="NAME", text="Peter W."),
         Span(type="LOCATION", text="Riverside General Hospital"),
         Span(type="DATE", text="June 3rd, 2023"),
         Span(type="ID", text="RGH-445120"),
     ])),
    # 2) tricky: nickname, title, relative + short dates, partial address, ID label, contact
    ("f/u for pt Elizabeth H. (goes by Liz) w/ hx of lupus, seen by Dr. Okafor last Tuesday at the "
     "Elm St. clinic in Portland, prev visit Mar 2nd '23. Acct#: PX-77120. call 503-555-0182 or "
     "email liz.h@mailbox.net. any interactions w/ hydroxychloroquine?",
     Extraction(spans=[
         Span(type="NAME", text="Elizabeth H."),
         Span(type="NAME", text="Liz"),
         Span(type="NAME", text="Dr. Okafor"),
         Span(type="DATE", text="last Tuesday"),
         Span(type="LOCATION", text="Elm St. clinic"),
         Span(type="LOCATION", text="Portland"),
         Span(type="DATE", text="Mar 2nd '23"),
         Span(type="ID", text="PX-77120"),
         Span(type="CONTACT", text="503-555-0182"),
         Span(type="CONTACT", text="liz.h@mailbox.net"),
     ])),
    # 3) hard negative: age, year on its own, eponyms
    ("Is a 91-year-old Hispanic woman diagnosed with Hashimoto's thyroiditis in 2020 still a candidate "
     "for the Whipple procedure according to the 2023 guidelines?",
     Extraction(spans=[])),
]

for example_text, example_output in FEWSHOT:
    for s in example_output.spans:
        assert s.text in example_text, f"few-shot span not verbatim: {s.text!r}"
print(len(FEWSHOT), "few-shot examples OK")

3 few-shot examples OK


### 4.3 Message builder and prompt fingerprint

Few-shot examples are added as earlier user/assistant turns, so the model sees the exact output format as its own previous answers.

`prompt_sha` is a short fingerprint of the system prompt plus examples. It is saved with every prediction, so after a prompt change you can tell which version produced which file.

In [8]:
def build_messages(text, fewshot):
    messages = [{"role": "system", "content": SYSTEM}]
    if fewshot:
        for example_text, example_output in FEWSHOT:
            messages.append({"role": "user", "content": example_text})
            messages.append({"role": "assistant", "content": example_output.model_dump_json()})
    messages.append({"role": "user", "content": text})
    return messages

def prompt_sha(fewshot):
    payload = json.dumps(build_messages("", fewshot), ensure_ascii=False)
    return hashlib.sha1(payload.encode()).hexdigest()[:8]

print("zero-shot prompt:", prompt_sha(False), "| few-shot prompt:", prompt_sha(True))
build_messages(data[0]["text"], fewshot=True)[-3:]

zero-shot prompt: 77323a06 | few-shot prompt: 9b9a935c


[{'role': 'user',
  'content': "Is a 91-year-old Hispanic woman diagnosed with Hashimoto's thyroiditis in 2020 still a candidate for the Whipple procedure according to the 2023 guidelines?"},
 {'role': 'assistant', 'content': '{"spans":[]}'},
 {'role': 'user',
  'content': 'What is the latest treatment protocol for a 34-year-old female diagnosed with MS like Anna S., previously treated at Methodist Hospital on April 12, 2023?'}]

## 5. Model and generation

The model runs inside the notebook with `transformers`; `outlines` enforces the JSON schema while decoding.

- **Non-thinking** (`enable_thinking=False`): greedy decoding, constrained to the schema from the first token. Qwen3's default generation config samples, so `do_sample=False` is set explicitly to keep runs deterministic.
- **Thinking** (`enable_thinking=True`): two stages, because a JSON constraint from the first token would block the `<think>` text.
  1. The reasoning is generated freely until `</think>`, with Qwen's recommended sampling settings (greedy decoding can make Thinking mode loop).
  2. The reasoning is added to the prompt, and the JSON answer is generated under the constraint.

The first cell checks which queries in the `RUN` conditions already have a prediction. The model is loaded only if something is missing.

In [ ]:
def pred_path(condition):
    return PRED_DIR / f"{MODEL_TAG}_{condition}.jsonl"

def records_for(condition):
    records = data[:LIMIT] if LIMIT else data
    if CONDITIONS[condition]["think"] and THINK_SUBSET:
        records = random.Random(SEED).sample(records, min(THINK_SUBSET, len(records)))
    return records

def todo_records(condition):
    """records of this condition that have no line in its prediction file yet"""
    done = set()
    if pred_path(condition).exists():
        with pred_path(condition).open(encoding="utf-8") as f:
            done = {json.loads(line)["id"] for line in f}
    return [r for r in records_for(condition) if r["id"] not in done]

TODO = {c: todo_records(c) for c in RUN}
for c, todo in TODO.items():
    print(f"{c}: {len(records_for(c)) - len(todo)} cached, {len(todo)} to run  ({pred_path(c)})")
NEED_MODEL = any(TODO.values())
print("load model:", NEED_MODEL)

In [ ]:
if NEED_MODEL:
    import torch
    import outlines
    from transformers import AutoTokenizer, AutoModelForCausalLM

    if torch.cuda.is_available():
        device, dtype = "cuda", torch.bfloat16
    elif torch.backends.mps.is_available():
        device, dtype = "mps", torch.float16
    else:
        device, dtype = "cpu", torch.float32

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    hf_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype).to(device)
    hf_model.eval()

    ol_model = outlines.from_transformers(hf_model, tokenizer)
    json_generator = outlines.Generator(ol_model, Extraction)   # schema is compiled once and reused

    THINK_END_ID = tokenizer.convert_tokens_to_ids("</think>")
    print(MODEL_ID, "on", device, dtype)
else:
    print("all RUN conditions are complete: model not loaded, existing prediction files are reused")

In [10]:
MAX_JSON_TOKENS  = 512
MAX_THINK_TOKENS = 4096

def render(messages, think):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=think)

def generate(messages, think, seed=SEED):
    """Returns (raw_json_output, thinking_text_or_None, thinking_truncated)."""
    prompt = render(messages, think)
    torch.manual_seed(seed)

    if not think:
        raw = json_generator(prompt, max_new_tokens=MAX_JSON_TOKENS, do_sample=False)
        return raw, None, False

    # stage 1: free reasoning, stop at </think>
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = hf_model.generate(**inputs, max_new_tokens=MAX_THINK_TOKENS, do_sample=True,
                                temperature=0.6, top_p=0.95, top_k=20,
                                eos_token_id=[THINK_END_ID, tokenizer.eos_token_id])
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    truncated = THINK_END_ID not in new_tokens.tolist()
    thinking = tokenizer.decode(new_tokens, skip_special_tokens=True)
    thinking = thinking.replace("<think>", "").replace("</think>", "").strip()

    # stage 2: constrained JSON, conditioned on the reasoning
    prompt2 = prompt + f"<think>\n{thinking}\n</think>\n\n"
    raw = json_generator(prompt2, max_new_tokens=MAX_JSON_TOKENS, do_sample=False)
    return raw, thinking, truncated

## 6. Post-processing

### 6.1 Parse

This turns the raw string into `{type, text}` dictionaries. Constrained decoding should mean it never fails; `parse_ok` records whether it did (e.g. if the output was cut off at `MAX_JSON_TOKENS`).

In [11]:
def parse(raw):
    try:
        extraction = Extraction.model_validate_json(raw)
        return [s.model_dump(mode="json") for s in extraction.spans], True
    except ValidationError as e:
        print("parse failure:", str(e)[:200])
        return [], False

parse('{"spans": [{"type": "NAME", "text": "Anna S."}]}')

([{'type': 'NAME', 'text': 'Anna S.'}], True)

### 6.2 Align predicted spans to character offsets

The model returns strings, but scoring works with positions.
- **Exact match first**, with a cursor per string: if the model lists "UCSF" twice, the entries map to the 1st and 2nd occurrences.
- **Fuzzy fallback:** case-insensitive, curly quotes treated as straight quotes, extra whitespace ignored. The match is mapped back to original offsets.
- **Not found:** kept with `grounded=False`. These are hallucinated spans and count as false positives.
- **Duplicates** (same offsets and label) are dropped.

In [12]:
def normalize_with_map(s):
    """lowercase, unify quotes, collapse whitespace; also return map normalized index -> original index"""
    chars, index_map, prev_space = [], [], False
    for i, ch in enumerate(s):
        ch = ch.translate(QUOTES).lower()
        if ch.isspace():
            if prev_space:
                continue
            ch, prev_space = " ", True
        else:
            prev_space = False
        chars.append(ch)
        index_map.append(i)
    return "".join(chars), index_map

def fuzzy_find(text, span, start=0):
    norm_text, index_map = normalize_with_map(text)
    norm_span, _ = normalize_with_map(span.strip())
    if not norm_span:
        return None
    j = norm_text.find(norm_span, bisect.bisect_left(index_map, start))
    if j < 0:
        return None
    return index_map[j], index_map[j + len(norm_span) - 1] + 1

def align(text, spans):
    cursor, seen, aligned = {}, set(), []
    for s in spans:
        key = s["text"]
        i = text.find(key, cursor.get(key, 0)) if key else -1
        if i >= 0:
            start, end, how = i, i + len(key), "exact"
        else:
            hit = fuzzy_find(text, key, cursor.get(key, 0)) or fuzzy_find(text, key)
            start, end, how = (hit[0], hit[1], "fuzzy") if hit else (None, None, "none")
        if start is not None:
            cursor[key] = end
            if (start, end, s["type"]) in seen:
                continue
            seen.add((start, end, s["type"]))
        aligned.append({**s, "start": start, "end": end, "grounded": start is not None, "match": how})
    return aligned

align("Bobby saw Bobby\u2019s  mother in TUCSON.",
      [{"type": "NAME", "text": "Bobby"}, {"type": "NAME", "text": "Bobby"},
       {"type": "LOCATION", "text": "Tucson"}, {"type": "NAME", "text": "Alice"}])

[{'type': 'NAME',
  'text': 'Bobby',
  'start': 0,
  'end': 5,
  'grounded': True,
  'match': 'exact'},
 {'type': 'NAME',
  'text': 'Bobby',
  'start': 10,
  'end': 15,
  'grounded': True,
  'match': 'exact'},
 {'type': 'LOCATION',
  'text': 'Tucson',
  'start': 29,
  'end': 35,
  'grounded': True,
  'match': 'fuzzy'},
 {'type': 'NAME',
  'text': 'Alice',
  'start': None,
  'end': None,
  'grounded': False,
  'match': 'none'}]

## 7. Run the conditions

Each condition writes one JSONL file with one line per query: the raw output, the thinking trace, the parsed and aligned spans, and metadata.

The run is **resumable**: IDs already in the file are skipped, and each line is flushed as soon as it is written.

> **If you change the prompt**, delete the old prediction files first and then re run, to not mix prompt versions.

In [ ]:
def run_condition(condition):
    cfg = CONDITIONS[condition]
    path = pred_path(condition)
    todo = todo_records(condition)
    print(f"{condition}: {len(records_for(condition)) - len(todo)} cached, {len(todo)} to run")
    if not todo:
        return path
    sha = prompt_sha(cfg["fewshot"])

    with path.open("a", encoding="utf-8") as f:
        for r in tqdm(todo, desc=condition):
            messages = build_messages(r["text"], cfg["fewshot"])
            t0 = time.time()
            raw, thinking, think_truncated = generate(messages, cfg["think"])
            spans, ok = parse(raw)
            row = {
                "id": r["id"], "condition": condition, "model": MODEL_ID,
                "prompt_sha": sha, "data_sha": DATA_SHA,
                "raw_output": raw, "thinking": thinking, "think_truncated": think_truncated,
                "parse_ok": ok, "spans": align(r["text"], spans),
                "latency_s": round(time.time() - t0, 2),
            }
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            f.flush()
    return path

### 7.1 Smoke test

Before a full run, check a few queries by eye: are span boundaries right, labels right, anything missing?

In [ ]:
if NEED_MODEL:
    for r in data[:5]:
        raw, _, _ = generate(build_messages(r["text"], fewshot=True), think=False)
        spans, ok = parse(raw)
        print("TEXT:", r["text"])
        print("PRED:", [(s["type"], s["text"], s["grounded"]) for s in align(r["text"], spans)])
        print("GOLD:", [(g["label"], g["text"]) for g in r["gold"]])
        print("-" * 100)
else:
    print("skipped: model not loaded (all RUN conditions are complete)")

### 7.2 Full runs

The thinking condition is much slower because of the reasoning traces. Time a few queries first, and if needed set `THINK_SUBSET` in Section 1. The conditions to run are listed in `RUN` (Section 1).

In [ ]:
for condition in RUN:
    print("wrote", run_condition(condition))

### 7.3 Non-LLM baseline: Presidio

Comparing zero-shot and few-shot only shows how prompting changes the model, so we add an existing de-identification tool as a reference point: [Microsoft Presidio](https://github.com/microsoft/presidio) (`presidio-analyzer`), which combines spaCy NER (`en_core_web_lg`) with rule-based recognizers (regex, checksums, context words). It is run with its default settings and no score threshold; each span keeps Presidio's own type and score.

Presidio's entity types are mapped onto the shared taxonomy; other types (e.g. `URL`, `NRP`) are dropped:

| Presidio type | shared label |
|---|---|
| `PERSON` | NAME |
| `DATE_TIME` | DATE |
| `LOCATION` | LOCATION |
| `PHONE_NUMBER`, `EMAIL_ADDRESS`, `IP_ADDRESS` | CONTACT |
| `US_SSN`, `MEDICAL_LICENSE`, `US_DRIVER_LICENSE`, `US_PASSPORT`, `US_BANK_NUMBER`, `US_ITIN`, `CREDIT_CARD`, `IBAN_CODE`, `UK_NHS` | ID |

Presidio returns character offsets, so no alignment step is needed. The output uses the same file format as the LLM conditions and is scored by the same code in Section 8. Like the LLM runs, it is resumable.

In [ ]:
PRESIDIO_TO_SHARED = {
    "PERSON": "NAME",
    "DATE_TIME": "DATE",
    "LOCATION": "LOCATION",
    "PHONE_NUMBER": "CONTACT", "EMAIL_ADDRESS": "CONTACT", "IP_ADDRESS": "CONTACT",
    "US_SSN": "ID", "MEDICAL_LICENSE": "ID", "US_DRIVER_LICENSE": "ID", "US_PASSPORT": "ID",
    "US_BANK_NUMBER": "ID", "US_ITIN": "ID", "CREDIT_CARD": "ID", "IBAN_CODE": "ID", "UK_NHS": "ID",
}

def presidio_spans(analyzer, text):
    spans, seen = [], set()
    for res in sorted(analyzer.analyze(text=text, language="en"), key=lambda r: (r.start, r.end)):
        label = PRESIDIO_TO_SHARED.get(res.entity_type)
        if label is None or (res.start, res.end, label) in seen:
            continue
        seen.add((res.start, res.end, label))
        spans.append({"type": label, "text": text[res.start:res.end], "start": res.start, "end": res.end,
                      "grounded": True, "match": "exact",
                      "presidio_type": res.entity_type, "score": round(res.score, 2)})
    return spans

In [ ]:
def run_presidio():
    path = pred_path("presidio")
    todo = todo_records("presidio")
    print(f"presidio: {len(records_for('presidio')) - len(todo)} cached, {len(todo)} to run")
    if not todo:
        return path

    from importlib.metadata import version
    from presidio_analyzer import AnalyzerEngine
    analyzer = AnalyzerEngine()   # default English setup: spaCy en_core_web_lg + built-in recognizers
    model = f"presidio-analyzer {version('presidio-analyzer')}"

    with path.open("a", encoding="utf-8") as f:
        for r in tqdm(todo, desc="presidio"):
            t0 = time.time()
            row = {
                "id": r["id"], "condition": "presidio", "model": model,
                "prompt_sha": "n/a", "data_sha": DATA_SHA,
                "parse_ok": True, "spans": presidio_spans(analyzer, r["text"]),
                "latency_s": round(time.time() - t0, 2),
            }
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            f.flush()
    return path

print("wrote", run_presidio())

### 7.4 Check the prediction files

Before scoring, each file should have: every expected ID exactly once, 100% `parse_ok`, and a single `prompt_sha`.

In [16]:
def load_predictions(condition):
    path = pred_path(condition)
    if not path.exists():
        return None
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f]

checks = []
for condition in CONDITIONS:
    rows = load_predictions(condition)
    if rows is None:
        continue
    expected = {r["id"] for r in records_for(condition)}
    ids = [row["id"] for row in rows]
    spans = [s for row in rows for s in row["spans"]]
    checks.append({
        "condition": condition,
        "rows": len(rows),
        "missing_ids": len(expected - set(ids)),
        "duplicate_ids": len(ids) - len(set(ids)),
        "parse_ok_%": round(100 * sum(row["parse_ok"] for row in rows) / len(rows), 2),
        "think_truncated": sum(row.get("think_truncated", False) for row in rows),
        "spans": len(spans),
        "ungrounded_%": round(100 * sum(not s["grounded"] for s in spans) / max(len(spans), 1), 2),
        "fuzzy_%": round(100 * sum(s["match"] == "fuzzy" for s in spans) / max(len(spans), 1), 2),
        "prompt_versions": sorted({row["prompt_sha"] for row in rows}),
        "mean_latency_s": round(sum(row["latency_s"] for row in rows) / len(rows), 2),
    })
pd.DataFrame(checks)

,condition,rows,missing_ids,duplicate_ids,parse_ok_%,think_truncated,spans,ungrounded_%,fuzzy_%,prompt_versions,mean_latency_s
0,zeroshot,1051,0,0,100.0,0,1166,23.07,0.43,[77323a06],1.67
1,fewshot,1051,0,0,100.0,0,1460,1.30,0.48,[9b9a935c],2.18


## 8. Evaluation

### 8.1 Matching

Every predicted span is matched against the gold spans of the same query in three modes. Each gold span can match at most one prediction:

| mode | a prediction matches a gold span if… | answers |
|---|---|---|
| `strict` | same start, end and label | exact extraction quality |
| `overlap` | characters overlap and same label | tolerant of boundary differences (e.g. `Dr. Lopez` vs `Lopez`, which ASQ itself annotates inconsistently) |
| `any` | characters overlap, label ignored | **privacy view:** would this PHI have been redacted? |

Ungrounded predictions (not found in the text) never match, so they count as false positives.

In [17]:
MODES = ["strict", "overlap", "any"]

def overlap_len(a, b):
    return max(0, min(a["end"], b["end"]) - max(a["start"], b["start"]))

def is_match(gold_span, pred_span, mode):
    if pred_span["start"] is None or gold_span["start"] is None:
        return False
    if mode == "strict":
        return (gold_span["start"], gold_span["end"], gold_span["label"]) == \
               (pred_span["start"], pred_span["end"], pred_span["type"])
    if overlap_len(gold_span, pred_span) == 0:
        return False
    return mode == "any" or gold_span["label"] == pred_span["type"]

def match_record(gold, preds, mode):
    """greedy one-to-one matching; returns {gold_index: pred_index}"""
    pairs, used = {}, set()
    for gi, g in enumerate(gold):
        candidates = [pi for pi, p in enumerate(preds) if pi not in used and is_match(g, p, mode)]
        if candidates:
            best = max(candidates, key=lambda pi: overlap_len(g, preds[pi]))
            pairs[gi] = best
            used.add(best)
    return pairs

### 8.2 Outcome table

This builds one row per gold span and per predicted span, with a `matched_<mode>` column for each mode. All metrics and the error analysis are computed from this table.

By default only **query IDs present in every prediction file** are evaluated, so conditions are compared on the same queries (this matters when thinking runs on a subset).

In [18]:
DATA_BY_ID = {r["id"]: r for r in data}
EVAL_ON_COMMON_IDS = True

predictions = {c: load_predictions(c) for c in CONDITIONS}
predictions = {c: rows for c, rows in predictions.items() if rows}
assert predictions, "no prediction files found: run Section 7 first"

common_ids = set.intersection(*[{row["id"] for row in rows} for rows in predictions.values()])
print("conditions:", list(predictions), "| common query IDs:", len(common_ids))

def build_outcomes(condition, rows):
    out = []
    for row in rows:
        if EVAL_ON_COMMON_IDS and row["id"] not in common_ids:
            continue
        record = DATA_BY_ID[row["id"]]
        gold, preds = record["gold"], row["spans"]
        pairs = {mode: match_record(gold, preds, mode) for mode in MODES}
        matched_preds = {mode: set(pairs[mode].values()) for mode in MODES}
        for gi, g in enumerate(gold):
            any_pi = pairs["any"].get(gi)
            out.append({
                "condition": condition, "id": row["id"], "side": "gold",
                "label": g["label"], "asq_type": g["asq_type"], "text": g["text"],
                "start": g["start"], "end": g["end"],
                **{f"matched_{m}": gi in pairs[m] for m in MODES},
                "any_pred_label": preds[any_pi]["type"] if any_pi is not None else None,
                "any_pred_text": preds[any_pi]["text"] if any_pi is not None else None,
            })
        for pi, p in enumerate(preds):
            out.append({
                "condition": condition, "id": row["id"], "side": "pred",
                "label": p["type"], "asq_type": None, "text": p["text"],
                "start": p["start"], "end": p["end"], "grounded": p["grounded"],
                **{f"matched_{m}": pi in matched_preds[m] for m in MODES},
            })
    return out

outcomes = pd.DataFrame([o for c, rows in predictions.items() for o in build_outcomes(c, rows)])
outcomes.to_csv(RESULTS_DIR / f"{MODEL_TAG}_outcomes.csv", index=False)
outcomes.head()

conditions: ['zeroshot', 'fewshot'] | common query IDs: 1051


,condition,id,side,label,asq_type,text,start,end,matched_strict,matched_overlap,matched_any,any_pred_label,any_pred_text,grounded
0,zeroshot,asq_0000,gold,NAME,NAME,Anna S.,86.0,93.0,False,False,False,NaN,NaN,NaN
1,zeroshot,asq_0000,gold,LOCATION,GEOGRAPHIC_LOCATION,Methodist Hospital,117.0,135.0,False,False,False,NaN,NaN,NaN
2,zeroshot,asq_0000,gold,DATE,DATE,"April 12, 2023",139.0,153.0,True,True,True,DATE,"April 12, 2023",NaN
3,zeroshot,asq_0000,pred,DATE,NaN,"April 12, 2023",139.0,153.0,True,True,True,NaN,NaN,True
4,zeroshot,asq_0001,gold,NAME,NAME,James T.,89.0,97.0,False,False,False,NaN,NaN,NaN


### 8.3 Overall precision, recall and F1

- **TP** = matched gold spans, **FN** = unmatched gold spans, **FP** = unmatched predicted spans.
- Scores are **micro-averaged** (pooled over all spans).
- **Recall is the primary metric**, because a missed PHI mention is a privacy leak.

In [19]:
def prf(frame, mode):
    gold = frame[frame.side == "gold"]
    pred = frame[frame.side == "pred"]
    tp = int(gold[f"matched_{mode}"].sum())
    fn = len(gold) - tp
    fp = int((~pred[f"matched_{mode}"]).sum())
    precision = tp / (tp + fp) if tp + fp else float("nan")
    recall = tp / (tp + fn) if tp + fn else float("nan")
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else float("nan")
    return {"TP": tp, "FP": fp, "FN": fn,
            "precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4)}

overall = pd.DataFrame([
    {"condition": c, "mode": m, **prf(outcomes[outcomes.condition == c], m)}
    for c in predictions for m in MODES
])
overall.to_csv(RESULTS_DIR / f"{MODEL_TAG}_overall.csv", index=False)
overall.pivot(index="condition", columns="mode", values=["recall", "precision", "f1"])

recall                 precision                      f1          \
mode          any overlap  strict       any overlap  strict     any overlap   
condition                                                                     
fewshot    0.4689  0.4268  0.3989    0.9548  0.8692  0.8123  0.6289  0.5725   
zeroshot   0.2879  0.2849  0.2674    0.7341  0.7264  0.6818  0.4136  0.4093   

                   
mode       strict  
condition          
fewshot    0.5351  
zeroshot   0.3842

### 8.4 Per-category results

An aggregate F1 score can hide rare PHI types, so results are also reported:
- per **shared category**: precision uses the predicted label, recall uses the gold label;
- per **original ASQ type**: recall only, because the model predicts shared labels. `n` shows how rare each type is; with 1–2 examples the numbers are anecdotal.

In [20]:
per_category = []
for c in predictions:
    frame = outcomes[outcomes.condition == c]
    for label in SHARED_TAXONOMY:
        for m in MODES:
            sub = frame[frame.label == label]
            per_category.append({"condition": c, "label": label, "mode": m,
                                 "n_gold": int((sub.side == "gold").sum()), **prf(sub, m)})
per_category = pd.DataFrame(per_category)
per_category.to_csv(RESULTS_DIR / f"{MODEL_TAG}_per_category.csv", index=False)

per_category[per_category["mode"] == "overlap"] \
    .pivot(index=["label", "n_gold"], columns="condition", values=["recall", "precision"])

recall          precision         
condition       fewshot zeroshot   fewshot zeroshot
label    n_gold                                    
CONTACT  79      0.3797   0.2025    0.8333   0.8421
DATE     806     0.6489   0.8263    0.9685   0.6952
ID       448     0.8728   0.1897    0.7405   0.8019
LOCATION 826     0.1077   0.0436    0.9780   0.9730
NAME     814     0.2899   0.0541    0.8906   0.9565

In [21]:
gold_outcomes = outcomes[outcomes.side == "gold"]
per_asq_type = (gold_outcomes
                .groupby(["asq_type", "condition"])
                .agg(n=("id", "size"),
                     recall_overlap=("matched_overlap", "mean"),
                     recall_any=("matched_any", "mean"))
                .round(4)
                .reset_index())
per_asq_type.to_csv(RESULTS_DIR / f"{MODEL_TAG}_per_asq_type.csv", index=False)
per_asq_type.pivot(index=["asq_type", "n"], columns="condition", values=["recall_overlap", "recall_any"]) \
    .sort_index(level="n", ascending=False)

recall_overlap          recall_any         
condition                                 fewshot zeroshot    fewshot zeroshot
asq_type                       n                                              
GEOGRAPHIC_LOCATION            826         0.1077   0.0436     0.1659   0.0436
NAME                           814         0.2899   0.0541     0.3600   0.0590
DATE                           806         0.6489   0.8263     0.6489   0.8263
MEDICAL_RECORD_NUMBER          305         0.9213   0.1869     0.9213   0.1869
HEALTH_PLAN_BENEFICIARY_NUMBER 91          0.7912   0.2637     0.7912   0.2857
PHONE_NUMBER                   45          0.4889   0.2889     0.6889   0.3333
SOCIAL_SECURITY_NUMBER         33          0.6364   0.0606     0.6667   0.0606
EMAIL_ADDRESS                  31          0.2258   0.0968     0.4839   0.1290
UNIQUE_IDENTIFIER              14          0.8571   0.1429     0.8571   0.1429
ACCOUNT_NUMBER                 4           1.0000   0.0000     1.0000   0.0000
FAX_NUMBER                     2           0.5000   0.0000     1.0000   0.0000
IP_ADDRESS                     1           0.0000   0.0000     1.0000   0.0000
CERTIFICATE_LICENSE_NUMBER     1           1.0000   0.0000     1.0000   0.0000

### 8.5 Query-level privacy metrics

- **leak-free rate:** share of queries with PHI where *every* gold span was caught (`any` mode). This is the share of queries that would be fully de-identified.
- **false alarm rate:** share of queries *without* PHI where the model still predicted something, i.e. text that would be over-redacted.

In [22]:
query_level = []
for c in predictions:
    frame = outcomes[outcomes.condition == c]
    evaluated_ids = {row["id"] for row in predictions[c] if not EVAL_ON_COMMON_IDS or row["id"] in common_ids}
    phi_ids = {i for i in evaluated_ids if DATA_BY_ID[i]["gold"]}
    no_phi_ids = evaluated_ids - phi_ids
    gold = frame[frame.side == "gold"]
    leaked_ids = set(gold.loc[~gold.matched_any, "id"])
    flagged_ids = set(frame.loc[frame.side == "pred", "id"])
    query_level.append({
        "condition": c,
        "queries_with_phi": len(phi_ids),
        "leak_free_rate": round(1 - len(leaked_ids & phi_ids) / max(len(phi_ids), 1), 4),
        "queries_without_phi": len(no_phi_ids),
        "false_alarm_rate": round(len(flagged_ids & no_phi_ids) / max(len(no_phi_ids), 1), 4),
    })
query_level = pd.DataFrame(query_level)
query_level.to_csv(RESULTS_DIR / f"{MODEL_TAG}_query_level.csv", index=False)
query_level

,condition,queries_with_phi,leak_free_rate,queries_without_phi,false_alarm_rate
0,zeroshot,832,0.0096,219,0.3014
1,fewshot,832,0.0745,219,0.0365


## 9. Error analysis

### 9.1 False negatives: missed PHI (privacy leaks)

Gold spans that no prediction overlapped at all (`any` mode), shown with the query for context.

In [23]:
ANALYSE = "fewshot" if "fewshot" in predictions else next(iter(predictions))

missed = gold_outcomes[(gold_outcomes.condition == ANALYSE) & ~gold_outcomes.matched_any].copy()
missed["query"] = missed["id"].map(lambda i: DATA_BY_ID[i]["text"])
print(f"[{ANALYSE}] missed PHI by category:", missed.label.value_counts().to_dict())

pd.set_option("display.max_colwidth", 200)
missed[["id", "asq_type", "text", "query"]].head(30)

[fewshot] missed PHI by category: {'LOCATION': 689, 'NAME': 521, 'DATE': 283, 'ID': 56, 'CONTACT': 30}


,id,asq_type,text,query
4139,asq_0000,NAME,Anna S.,"What is the latest treatment protocol for a 34-year-old female diagnosed with MS like Anna S., previously treated at Methodist Hospital on April 12, 2023?"
4140,asq_0000,GEOGRAPHIC_LOCATION,Methodist Hospital,"What is the latest treatment protocol for a 34-year-old female diagnosed with MS like Anna S., previously treated at Methodist Hospital on April 12, 2023?"
4141,asq_0000,DATE,"April 12, 2023","What is the latest treatment protocol for a 34-year-old female diagnosed with MS like Anna S., previously treated at Methodist Hospital on April 12, 2023?"
4142,asq_0001,NAME,James T.,"Evaluation of long-term outcomes for bypass surgery in patients over 60, referencing Mr. James T., operated at St. Vincent's on May 30th, 2022 (ID: 987654321)?"
4143,asq_0001,GEOGRAPHIC_LOCATION,St. Vincent's,"Evaluation of long-term outcomes for bypass surgery in patients over 60, referencing Mr. James T., operated at St. Vincent's on May 30th, 2022 (ID: 987654321)?"
4144,asq_0001,DATE,"May 30th, 2022","Evaluation of long-term outcomes for bypass surgery in patients over 60, referencing Mr. James T., operated at St. Vincent's on May 30th, 2022 (ID: 987654321)?"
4147,asq_0003,NAME,John L.,"Rec mgmt of 70yo M w/ CHF, seen by Dr. John L. at Mt. Sinai on Feb 21, 2023. looking for alt tx options after beta-blocker intol."
4148,asq_0003,GEOGRAPHIC_LOCATION,Mt. Sinai,"Rec mgmt of 70yo M w/ CHF, seen by Dr. John L. at Mt. Sinai on Feb 21, 2023. looking for alt tx options after beta-blocker intol."
4152,asq_0004,NAME,Sarah P.,"Best practices for managing GERD in a 50-year-old woman, Sarah P., especially post-hysterectomy procedure done at UCLA Medical Center on March 5th, 2021?"
4158,asq_0005,GEOGRAPHIC_LOCATION,Elm Clinic,"Can I prescribe lisinopril to a hypertensive patient with a creatinine level of 2.1, David S., evaluated at Elm Clinic on Jan 15th, 2023 (MRN: 998877)?"


### 9.2 False positives: spurious predictions

Predictions that overlap no gold span. `grounded=False` means the text does not appear in the query (a hallucination). The most frequent strings show systematic over-tagging, e.g. ages, years or relative dates that ASQ did not annotate.

In [24]:
pred_outcomes = outcomes[outcomes.side == "pred"]
spurious = pred_outcomes[(pred_outcomes.condition == ANALYSE) & ~pred_outcomes.matched_any].copy()
spurious["query"] = spurious["id"].map(lambda i: DATA_BY_ID[i]["text"])

print(f"[{ANALYSE}] spurious predictions by label:", spurious.label.value_counts().to_dict())
print(f"[{ANALYSE}] hallucinated (not in text):", int((~spurious.grounded.astype(bool)).sum()))
display(spurious.groupby(["label", "text"]).size().sort_values(ascending=False).head(20).rename("count"))
spurious[["id", "label", "text", "grounded", "query"]].head(30)

[fewshot] spurious predictions by label: {'NAME': 28, 'DATE': 17, 'ID': 16, 'LOCATION': 4, 'CONTACT': 1}
[fewshot] hallucinated (not in text): 19


label     text               
NAME      Pt.                    4
          72-year-old female     3
ID        RGH-445120             3
DATE      June 3rd, 2023         3
NAME      female                 2
          62-year-old male       2
DATE      2021                   2
NAME      Lupus                  1
          80-year-old female     1
LOCATION  San Francisco          1
          St. Luke’s, Ward 7B    1
NAME      30-year-old female     1
          32-year-old female     1
          45yo male              1
          50yo M.                1
          55 yo F                1
          55-year-old female     1
          55-year-old male       1
          59yo male              1
          5yo                    1
Name: count, dtype: int64

,id,label,text,grounded,query
4151,asq_0003,ID,MTS-2023,False,"Rec mgmt of 70yo M w/ CHF, seen by Dr. John L. at Mt. Sinai on Feb 21, 2023. looking for alt tx options after beta-blocker intol."
4260,asq_0029,NAME,59yo male,True,"tx protocol for 59yo male w/ hx of HTN & DM, observed at John's Hopkins on Feb 1, 2023 (MRN: JH-123456). pt reportd rxns to Lisinopril."
4264,asq_0029,ID,MRN: JH-123456,True,"tx protocol for 59yo male w/ hx of HTN & DM, observed at John's Hopkins on Feb 1, 2023 (MRN: JH-123456). pt reportd rxns to Lisinopril."
4304,asq_0038,NAME,Pt w/ CAD & hx of MI,True,"rec. mx for pt w/ CAD & hx of MI, reviewed by Dr. Smith at Baylor Med. Center on Feb 2nd 2021. considering Plavix. Contact info is smith.example@baylor.edu."
4467,asq_0077,NAME,72-year-old female,True,"Latest treatment options for a 72-year-old female with Alzheimer's disease, referred to by Dr. John Lee at Nevada Medical Group on January 5, 2023 (Contact: 702-555-1234)?"
4511,asq_0088,NAME,80-year-old female,True,"Recommendations for anticoagulation therapy in an 80-year-old female with afib, seen at Elm Street Clinic on September 10th. Patient's insurance number: 123456789."
4667,asq_0125,NAME,55-year-old female,True,"Management of a 55-year-old female with chronic atrial fibrillation, seen by Dr. Smith at Cedars-Sinai, Los Angeles on August 12th, 2023 (MRN: 123-456-789)?"
4823,asq_0163,DATE,12/11/1958,True,"best practices for managing T2DM in pt seen by Dr. Stevens at the Downtown Clinic; DOB 12/11/1958, case reviewed Mar 3, 2023."
4908,asq_0187,LOCATION,San Francisco,False,"Need guidelines for handling a 50yo pt w/ hx of MI & HBP seen by Dr. Roberts in San Fran on Feb 2nd 2021, worried about statin interaction?"
4959,asq_0200,NAME,62-year-old male,True,"What are the treatment options for a 62-year-old male with chronic atrial fibrillation who was admitted to St. Mary's Hospital on April 12th, 2022, under Dr. Alan S.?"


### 9.3 Boundary errors and label confusions

- **Boundary errors:** the right label, but different start/end (matched in `overlap`, not in `strict`).
- **Label confusions:** the PHI was found, but with the wrong category (matched in `any`, not in `overlap`).

In [25]:
g = gold_outcomes[gold_outcomes.condition == ANALYSE]

boundary = g[g.matched_overlap & ~g.matched_strict]
print(f"[{ANALYSE}] boundary errors: {len(boundary)}")
display(boundary[["id", "label", "text", "any_pred_text"]].head(20))

confused = g[g.matched_any & ~g.matched_overlap]
print(f"[{ANALYSE}] label confusions: {len(confused)}")
pd.crosstab(confused.label, confused.any_pred_label, rownames=["gold"], colnames=["predicted"])

[fewshot] boundary errors: 83


,id,label,text,any_pred_text
4160,asq_0005,ID,998877,MRN: 998877
4204,asq_0015,ID,12345-JS,MRN 12345-JS
4290,asq_0035,ID,Patient ID: ABCD1234,ABCD1234
4366,asq_0054,ID,#DB-2345678,DB-2345678
4436,asq_0072,ID,123-45-6789,SSN: 123-45-6789
4471,asq_0079,NAME,Emily C.,Dr. Emily C.
4482,asq_0082,ID,patient ID: 897-65-4321,897-65-4321
4517,asq_0089,ID,#EM-345678,EM-345678
4528,asq_0091,ID,8765-4321,MRN 8765-4321
4631,asq_0117,ID,#R-987654,R-987654


[fewshot] label confusions: 125


predicted,CONTACT,ID,NAME
gold,,,
CONTACT,0,18,1
ID,1,0,0
LOCATION,1,47,0
NAME,3,54,0


### 9.4 Thinking ablation: which PHI does reasoning recover or lose?

This compares `fewshot` with `fewshot_think` on the same queries: gold spans caught with thinking but missed without (**recovered**), and the reverse (**lost**).

In [26]:
if {"fewshot", "fewshot_think"} <= set(predictions):
    key = ["id", "start", "end", "label"]
    a = gold_outcomes[gold_outcomes.condition == "fewshot"].set_index(key)["matched_any"]
    b = gold_outcomes[gold_outcomes.condition == "fewshot_think"].set_index(key)["matched_any"]
    both = pd.concat([a.rename("fewshot"), b.rename("think")], axis=1).dropna()
    recovered = both[~both.fewshot.astype(bool) & both.think.astype(bool)].reset_index()
    lost = both[both.fewshot.astype(bool) & ~both.think.astype(bool)].reset_index()
    print("recovered by thinking:", len(recovered), recovered.label.value_counts().to_dict())
    print("lost with thinking:   ", len(lost), lost.label.value_counts().to_dict())
    display(recovered.head(20))
else:
    print("run both 'fewshot' and 'fewshot_think' to compare")

run both 'fewshot' and 'fewshot_think' to compare
